# ThreeLetters

[Codility task](https://app.codility.com/programmers/trainings/5/three_letters/) - Exercise 5, Coding skills - *medium*

`solution(A, B)` returns a string with exactly `A` letters `'a'` and `B` letters `'b'` in which neither `"aaa"` nor `"bbb"` occurs. Any valid string is accepted.

| A | B | a valid answer |
|---|---|----------------|
| 5 | 3 | `aabaabab` |
| 3 | 3 | `ababab` |
| 1 | 4 | `bbabb` (the only one) |

Assumptions: `A`, `B` in `[0..100]` and a solution exists.

## Idea: greedy

Build the string one letter at a time:

- normally place the letter that has **more copies left** (so the scarcer letter is not wasted too early);
- but if the last two letters are equal, placing that letter again would make three in a row, so place the **other** one.

## When does a solution exist?

The `B` letters `b` leave `B + 1` slots for runs of `a`, each of length at most 2, so `A <= 2 * (B + 1)`. Symmetrically, `B <= 2 * (A + 1)`. The task guarantees this; the function raises `ValueError` otherwise, which makes misuse visible.

In [ ]:
def solvable(A, B):
    return A <= 2 * (B + 1) and B <= 2 * (A + 1)


def solution(A, B):
    if not solvable(A, B):
        raise ValueError(f"no valid string for A={A}, B={B}")
    remaining = {"a": A, "b": B}
    result = []
    while remaining["a"] or remaining["b"]:
        if len(result) >= 2 and result[-1] == result[-2]:
            letter = "b" if result[-1] == "a" else "a"        # forced: avoid three in a row
        else:
            letter = "a" if remaining["a"] >= remaining["b"] else "b"
        remaining[letter] -= 1
        result.append(letter)
    return "".join(result)

## Validator

Many strings are correct, so tests check **properties**: the letter counts, the length and the absence of `"aaa"` / `"bbb"`.

In [ ]:
def is_valid(s, A, B):
    return (
        set(s) <= {"a", "b"}
        and s.count("a") == A
        and s.count("b") == B
        and "aaa" not in s
        and "bbb" not in s
    )


assert is_valid("aabaabab", 5, 3) and is_valid("abaabbaa", 5, 3)
assert not is_valid("aaabbb", 3, 3)             # three in a row
assert not is_valid("abab", 3, 1)               # wrong counts
print("validator OK")

## Official examples

In [ ]:
for A, B in [(5, 3), (3, 3)]:
    s = solution(A, B)
    print(f"A={A}, B={B} ->", s)
    assert is_valid(s, A, B)

assert solution(1, 4) == "bbabb"                 # the only correct answer
print("examples OK")

## Edge cases

In [ ]:
assert solution(0, 0) == ""
assert solution(0, 1) == "b"
assert solution(1, 0) == "a"
assert solution(0, 2) == "bb"
assert solution(2, 0) == "aa"
assert solution(4, 1) == "aabaa"                 # the mirror of the (1, 4) example
for A, B in [(100, 100), (100, 49), (49, 100), (100, 50), (50, 100), (1, 2), (2, 1)]:
    assert is_valid(solution(A, B), A, B), (A, B)

for A, B in [(3, 0), (0, 3), (1, 5), (5, 1), (100, 0)]:
    try:
        solution(A, B)
    except ValueError:
        pass
    else:
        raise AssertionError(f"({A}, {B}) has no solution but no error was raised")
print("edge cases OK")

## Exhaustive checks

1. **Existence rule vs brute force.** For every string of length up to 12 over `{a, b}`, record which `(A, B)` pairs have at least one valid string, and compare with `solvable`.
2. **Greedy vs every pair in range.** For all `(A, B)` in `[0..100]^2` that are solvable, the greedy result must be valid (5,301 of the 10,201 pairs are solvable); the others must raise `ValueError`.

In [ ]:
import itertools

# 1. which (A, B) really have a valid string? (all strings up to length 12)
achievable = set()
for n in range(0, 13):
    for letters in itertools.product("ab", repeat=n):
        s = "".join(letters)
        if "aaa" not in s and "bbb" not in s:
            achievable.add((s.count("a"), s.count("b")))

for A in range(13):
    for B in range(13 - A):
        assert ((A, B) in achievable) == solvable(A, B), (A, B)
print("existence rule matches brute force for A + B <= 12")

# 2. greedy on the whole input range
solved = 0
for A in range(101):
    for B in range(101):
        if solvable(A, B):
            assert is_valid(solution(A, B), A, B), (A, B)
            solved += 1
        else:
            try:
                solution(A, B)
                raise AssertionError((A, B))
            except ValueError:
                pass
print(f"greedy produced a valid string for all {solved} solvable pairs in [0..100]^2")

## Notes

- Tie rule: when `A == B` the code starts with `'a'`; any consistent choice works.
- Why greedy is safe: taking the more frequent letter keeps the counts balanced enough that the `A <= 2(B+1)` / `B <= 2(A+1)` invariant survives each step (checked exhaustively above rather than proved here).
- `remaining` as a dict keeps the two letters symmetric and avoids duplicating branches.
- The function is O(A + B); at most 200 letters here.